# ETL SINISA – Diagnóstico e Limpeza

**Squad Observatório do Clima – Grande Desafio DataConnect Solutions**

Este notebook é o script completo de ETL (extração, padronização e exportação)
que localiza, limpa e junta, a partir dos três arquivos brutos do SINISA
(Ministério das Cidades), as informações que a squad usa no painel de
priorização de municípios do Carlos (Sedec/MIDR):

1. Quantidade de domicílios em área de risco de inundação (arquivo 1)
2. Se existe mapeamento de área de risco (arquivo 1)
3. Se o município é classificado como "crítico" (arquivo 2)
4. Percentual de vias com rede de drenagem (arquivo 2)
5. Investimento per capita em drenagem (arquivo 2)

O notebook segue quatro partes, nesta ordem: **(1)** diagnóstico dinâmico de
aba/linha/coluna nos 3 arquivos, cruzando os códigos com o glossário em PDF;
**(2)** limpeza estrutural — extração das colunas confirmadas, padronização
do `Cod_IBGE_7`, conversão de tipos e junção dos dois arquivos por município;
**(3)** documentação (log de transformações); **(4)** exportação do
`sinisa_limpo.csv`.

**Antes de rodar:** tenha em mãos os 3 arquivos originais:
- `SINISA_AGUASPLUVIAIS_Informacoes_Formularios_Tecnicos.xlsx`
- `SINISA_AGUASPLUVIAIS_Indicadores_AP2024.xlsx`
- `GLOSSARIO_INFORMACOES_SINISA_AGUASPLUVIAIS.pdf`

## 0. Configuração do ambiente e upload dos arquivos

In [1]:
!pip install openpyxl pypdf -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 6.4 MB/s eta 0:00:00


In [2]:
import glob
import re
import pandas as pd
from openpyxl import load_workbook

pd.set_option("display.max_colwidth", 120)

# Faz upload manual se estiver rodando no Colab; se não, assume que os arquivos
# já estão na pasta atual (por exemplo, se você montou o Google Drive).
try:
    from google.colab import files
    print("Ambiente Colab detectado. Selecione os 3 arquivos originais no diálogo abaixo:")
    uploaded = files.upload()
except ImportError:
    print("Ambiente não é o Colab: assumindo que os arquivos já estão no diretório atual.")

def encontrar_arquivo(padrao):
    candidatos = glob.glob(padrao)
    if not candidatos:
        raise FileNotFoundError(
            f"Não encontrei nenhum arquivo correspondente a '{padrao}'. "
            "Confirme se o upload foi feito e se o nome do arquivo não mudou."
        )
    if len(candidatos) > 1:
        print(f"Aviso: mais de um arquivo bate com '{padrao}': {candidatos}. Usando o primeiro.")
    return candidatos[0]

CAMINHO_FORMULARIOS = encontrar_arquivo("*Formularios_Tecnicos*.xlsx")
CAMINHO_INDICADORES = encontrar_arquivo("*Indicadores_AP2024*.xlsx")
CAMINHO_GLOSSARIO = encontrar_arquivo("*GLOSSARIO*SINISA*.pdf")

print("Arquivo 1 (formulários técnicos):", CAMINHO_FORMULARIOS)
print("Arquivo 2 (indicadores):", CAMINHO_INDICADORES)
print("Glossário (PDF):", CAMINHO_GLOSSARIO)

Ambiente Colab detectado. Selecione os 3 arquivos originais no diálogo abaixo:


Saving GLOSSARIO_INFORMACOES_SINISA_AGUASPLUVIAIS.pdf to GLOSSARIO_INFORMACOES_SINISA_AGUASPLUVIAIS.pdf
Saving SINISA_AGUASPLUVIAIS_Indicadores_AP2024.xlsx to SINISA_AGUASPLUVIAIS_Indicadores_AP2024.xlsx
Saving SINISA_AGUASPLUVIAIS_Informacoes_Formularios_Tecnicos.xlsx to SINISA_AGUASPLUVIAIS_Informacoes_Formularios_Tecnicos.xlsx
Arquivo 1 (formulários técnicos): SINISA_AGUASPLUVIAIS_Informacoes_Formularios_Tecnicos.xlsx
Arquivo 2 (indicadores): SINISA_AGUASPLUVIAIS_Indicadores_AP2024.xlsx
Glossário (PDF): GLOSSARIO_INFORMACOES_SINISA_AGUASPLUVIAIS.pdf


## 1. Funções auxiliares de diagnóstico

Essas planilhas do SINISA têm um cabeçalho "empilhado": título do ministério,
título do sistema, título do módulo, data de geração, uma linha em branco, e só
então o cabeçalho real de colunas — seguido por linhas de fórmula/unidade/código
antes dos dados começarem de fato. Em vez de assumir um número fixo de linhas,
detectamos isso procurando marcadores conhecidos.

In [3]:
def carregar_aba_bruta(caminho_arquivo, nome_aba):
    """Carrega uma aba inteira sem cabeçalho, célula por célula, como strings/valores brutos."""
    wb = load_workbook(caminho_arquivo, read_only=True, data_only=True)
    ws = wb[nome_aba]
    dados = list(ws.iter_rows(values_only=True))
    return pd.DataFrame(dados)

def localizar_linha_cabecalho(df_bruto, marcador="Código IBGE", coluna=0, limite_busca=20):
    """Retorna o índice (0-based) da linha onde o cabeçalho real de colunas começa."""
    for i in range(min(limite_busca, len(df_bruto))):
        valor = df_bruto.iloc[i, coluna]
        if isinstance(valor, str) and valor.strip() == marcador:
            return i
    raise ValueError(f"Não encontrei a linha de cabeçalho (marcador '{marcador}' na coluna {coluna}).")

def localizar_inicio_dos_dados(df_bruto, coluna=0, a_partir_de=0):
    """Retorna o índice (0-based) da primeira linha cujo valor na coluna indicada
    parece um código IBGE de município (inteiro de 7 dígitos)."""
    for i in range(a_partir_de, len(df_bruto)):
        valor = df_bruto.iloc[i, coluna]
        if isinstance(valor, (int, float)) and not pd.isna(valor):
            if len(str(int(valor))) == 7:
                return i
    raise ValueError("Não encontrei o início dos dados (nenhum código IBGE de 7 dígitos encontrado).")

def montar_amostra(df_bruto, linha_cabecalho, linha_inicio_dados, n_linhas=5):
    """Aplica o cabeçalho encontrado e devolve as n_linhas primeiras linhas de dados reais."""
    nomes_colunas = df_bruto.iloc[linha_cabecalho].tolist()
    amostra = df_bruto.iloc[linha_inicio_dados: linha_inicio_dados + n_linhas].copy()
    amostra.columns = nomes_colunas
    return amostra.reset_index(drop=True)

## 2. Arquivo 1 — Formulários Técnicos

### 2.1 Abas disponíveis

In [4]:
wb1 = load_workbook(CAMINHO_FORMULARIOS, read_only=True)
print("Abas encontradas no arquivo 1:", wb1.sheetnames)

Abas encontradas no arquivo 1: ['Dados Operacionais', 'Dados sobre Gestão de Riscos']


### 2.2 Localizando as informações-alvo

Pelo nome das abas, a aba **"Dados sobre Gestão de Riscos"** é a candidata a
conter tanto "existe mapeamento de área de risco" quanto "quantidade de
domicílios em risco de inundação" — vamos confirmar isso vasculhando os nomes
reais de coluna, não assumindo pelo nome da aba.

In [5]:
ABA_RISCOS = "Dados sobre Gestão de Riscos"

bruto_riscos = carregar_aba_bruta(CAMINHO_FORMULARIOS, ABA_RISCOS)
linha_cab_riscos = localizar_linha_cabecalho(bruto_riscos)
linha_dados_riscos = localizar_inicio_dos_dados(bruto_riscos, a_partir_de=linha_cab_riscos)

print(f"Aba '{ABA_RISCOS}':")
print(f"  -> Cabeçalho de colunas na linha {linha_cab_riscos} (0-based) = linha {linha_cab_riscos + 1} do Excel")
print(f"  -> Dados começam na linha {linha_dados_riscos} (0-based) = linha {linha_dados_riscos + 1} do Excel")
print(f"  -> Total de linhas na aba: {len(bruto_riscos)}")

Aba 'Dados sobre Gestão de Riscos':
  -> Cabeçalho de colunas na linha 7 (0-based) = linha 8 do Excel
  -> Dados começam na linha 11 (0-based) = linha 12 do Excel
  -> Total de linhas na aba: 5581


In [6]:
nomes_colunas_riscos = bruto_riscos.iloc[linha_cab_riscos].tolist()

# Busca por substring nos nomes de coluna reais - sem assumir o texto exato de antemão
def buscar_colunas(nomes_colunas, *termos):
    encontrados = []
    for idx, nome in enumerate(nomes_colunas):
        if isinstance(nome, str) and all(t.lower() in nome.lower() for t in termos):
            encontrados.append((idx, nome))
    return encontrados

candidatas_mapeamento = buscar_colunas(nomes_colunas_riscos, "mapeamento", "risco")
candidatas_quantidade = buscar_colunas(nomes_colunas_riscos, "quantidade", "domic", "risco", "inunda")
candidatas_critico = buscar_colunas(nomes_colunas_riscos, "crítico")

print("Colunas candidatas para 'existe mapeamento de área de risco':")
for idx, nome in candidatas_mapeamento:
    print(f"  col {idx}: {nome!r}")

print("\nColunas candidatas para 'quantidade de domicílios em risco de inundação':")
for idx, nome in candidatas_quantidade:
    print(f"  col {idx}: {nome!r}")

print("\nColunas candidatas para 'município crítico':")
for idx, nome in candidatas_critico:
    print(f"  col {idx}: {nome!r}")

Colunas candidatas para 'existe mapeamento de área de risco':
  col 34: "Existe mapeamento de áreas de risco de inundação dos cursos d'água urbanos? *"
  col 46: 'O município possui mapeamento de risco geológico-geotécnico?'

Colunas candidatas para 'quantidade de domicílios em risco de inundação':
  col 38: 'Quantidade de domicílios sujeitos a risco de inundação segundo o Projeto de Setorização de Áreas em Alto e Muito Alto Risco a Movimentos de Massa, Enchente e Inundação (Fonte: CPRM)'
  col 39: 'Quantidade de domicílios sujeitos a risco de inundação. *'
  col 40: 'A quantidade de domicílios sujeitos a risco de inundação é diferente da informada na coleta passada. Justifique essa variação: *'
  col 41: 'Metodologia de determinação da quantidade de domicílios sujeitos a risco de inundação. *'
  col 42: 'Especifique qual a outra metodologia de determinação da quantidade de domicílios sujeitos a risco de inundação. *'
  col 43: 'Quantidade de domicílios em risco de inundação coberta pe

Também precisamos do código de campo do SINISA (ex.: `GAP2105`) de cada coluna
candidata, para depois cruzar com o glossário em PDF. Os códigos ficam numa
linha própria, entre o cabeçalho e o início dos dados.

In [7]:
# A linha de códigos fica a 3 linhas do cabeçalho, mas confirmamos procurando
# uma linha que contenha strings no padrão GAPxxxx / GAPxxxx* entre o cabeçalho e os dados
def localizar_linha_de_codigos(df_bruto, linha_cabecalho, linha_dados):
    padrao_codigo = re.compile(r"^[A-Z]{2,4}\d{3,4}\*?$")
    for i in range(linha_cabecalho + 1, linha_dados):
        valores = df_bruto.iloc[i].tolist()
        acertos = sum(1 for v in valores if isinstance(v, str) and padrao_codigo.match(v.strip()))
        if acertos >= 3:
            return i
    return None

linha_codigos_riscos = localizar_linha_de_codigos(bruto_riscos, linha_cab_riscos, linha_dados_riscos)
print(f"Linha de códigos de campo detectada: índice {linha_codigos_riscos} (0-based) = linha {linha_codigos_riscos + 1} do Excel\n")

codigos_riscos = bruto_riscos.iloc[linha_codigos_riscos].tolist() if linha_codigos_riscos is not None else None

for etiqueta, candidatas in [
    ("mapeamento de área de risco", candidatas_mapeamento),
    ("quantidade de domicílios em risco", candidatas_quantidade),
    ("crítico", candidatas_critico),
]:
    print(f"--- {etiqueta} ---")
    for idx, nome in candidatas:
        codigo = codigos_riscos[idx] if codigos_riscos is not None else "?"
        print(f"  col {idx} | código: {codigo!r} | nome completo: {nome!r}")
    print()

Linha de códigos de campo detectada: índice 10 (0-based) = linha 11 do Excel

--- mapeamento de área de risco ---
  col 34 | código: 'GAP2105*' | nome completo: "Existe mapeamento de áreas de risco de inundação dos cursos d'água urbanos? *"
  col 46 | código: 'GAP2116' | nome completo: 'O município possui mapeamento de risco geológico-geotécnico?'

--- quantidade de domicílios em risco ---
  col 38 | código: 'GAP2109' | nome completo: 'Quantidade de domicílios sujeitos a risco de inundação segundo o Projeto de Setorização de Áreas em Alto e Muito Alto Risco a Movimentos de Massa, Enchente e Inundação (Fonte: CPRM)'
  col 39 | código: 'GAP2110*' | nome completo: 'Quantidade de domicílios sujeitos a risco de inundação. *'
  col 40 | código: 'GAP2111*' | nome completo: 'A quantidade de domicílios sujeitos a risco de inundação é diferente da informada na coleta passada. Justifique essa variação: *'
  col 41 | código: 'GAP2112*' | nome completo: 'Metodologia de determinação da quantidade de

### 2.3 Amostra de 5 linhas — Arquivo 1 (aba "Dados sobre Gestão de Riscos")

Mostrando as colunas de identificação do município + as colunas-alvo encontradas acima.

In [8]:
colunas_interesse_riscos = [0, 1, 2, 5] + [idx for idx, _ in candidatas_mapeamento] + [idx for idx, _ in candidatas_quantidade]
colunas_interesse_riscos = sorted(set(colunas_interesse_riscos))

amostra_riscos = montar_amostra(bruto_riscos, linha_cab_riscos, linha_dados_riscos, n_linhas=5)
amostra_riscos.iloc[:, colunas_interesse_riscos]

,Código IBGE,Nome,UF,Crítico,Existe mapeamento de áreas de risco de inundação dos cursos d'água urbanos? *,"Quantidade de domicílios sujeitos a risco de inundação segundo o Projeto de Setorização de Áreas em Alto e Muito Alto Risco a Movimentos de Massa, Enchente e Inundação (Fonte: CPRM)",Quantidade de domicílios sujeitos a risco de inundação. *,A quantidade de domicílios sujeitos a risco de inundação é diferente da informada na coleta passada. Justifique essa variação: *,Metodologia de determinação da quantidade de domicílios sujeitos a risco de inundação. *,Especifique qual a outra metodologia de determinação da quantidade de domicílios sujeitos a risco de inundação. *,Quantidade de domicílios em risco de inundação coberta pelos sistemas de alerta em funcionamento. *,O município possui mapeamento de risco geológico-geotécnico?
0,1100015,Alta Floresta D Oeste,RO,Não,Sim,,150,,,,,
1,1100023,Ariquemes,RO,Sim,Sim,,53,,,,,
2,1100031,Cabixi,RO,Não,Não,,0,,,,,
3,1100049,Cacoal,RO,Sim,Sim,,430,,,,,
4,1100056,Cerejeiras,RO,Não,Não,,0,,,,,


### 2.4 Cruzando com o glossário oficial (PDF)

Extraindo o texto do PDF e procurando pela definição oficial de cada código
encontrado, para confirmar que a coluna significa o que o nome sugere — sem
assumir nada só pelo nome da coluna na planilha.

In [9]:
from pypdf import PdfReader

leitor = PdfReader(CAMINHO_GLOSSARIO)
texto_glossario = "\n".join(pagina.extract_text() or "" for pagina in leitor.pages)

def mostrar_definicao(codigo, texto_glossario, n_linhas=6):
    codigo_base = codigo.strip("*")
    pos = texto_glossario.find(codigo_base)
    if pos == -1:
        print(f"[{codigo_base}] NÃO encontrado no glossário — não assuma o significado, confirme manualmente.")
        return
    trecho = texto_glossario[pos: pos + 500]
    linhas = [l for l in trecho.split("\n") if l.strip()][:n_linhas]
    print(f"[{codigo_base}] " + " ".join(linhas))
    print()

codigos_para_checar = set()
for _, candidatas in [(None, candidatas_mapeamento), (None, candidatas_quantidade), (None, candidatas_critico)]:
    for idx, _ in candidatas:
        if codigos_riscos is not None and isinstance(codigos_riscos[idx], str):
            codigos_para_checar.add(codigos_riscos[idx])

for codigo in sorted(codigos_para_checar):
    mostrar_definicao(codigo, texto_glossario)

[Critico] NÃO encontrado no glossário — não assuma o significado, confirme manualmente.
[GAP2105] GAP2105 - O município possui mapeamento de áreas de risco de inundação na área urbana? Informar se já foi feito ou não, até o ﬁnal do ano de referência, o mapeamento das áreas de risco de inundação na área urbana do município. Bloco: 2 - Mapeamento de Áreas de Risco Referência: RI009  |  GAP2106 - O mapeamento é parcial ou integral? Indicar se o mapeamento de áreas de risco de inundação abrange a totalidade das áreas urbanas do município (mapeamento integral) ou apenas parte delas

[GAP2109] GAP2109 - Quantidade de domicílios sujeitos a risco de inundação (Fonte: CPRM): (Informação Futura) Quantidade de domicílios urbanos existentes no município, até o ﬁnal do ano de referência, suscetíveis a riscos de inundação, tendo ou não sido atingidos por eventos hidrológicos impactantes, segundo o Projeto de Setorização de Áreas em Alto e Muito Alto Risco a Movimentos de Massa, Enchente e Inundação 

## 3. Arquivo 2 — Indicadores AP2024

### 3.1 Abas disponíveis

In [10]:
wb2 = load_workbook(CAMINHO_INDICADORES, read_only=True)
print("Abas encontradas no arquivo 2:", wb2.sheetnames)

Abas encontradas no arquivo 2: ['Indicadores por município']


In [11]:
ABA_INDICADORES = wb2.sheetnames[0]

bruto_indicadores = carregar_aba_bruta(CAMINHO_INDICADORES, ABA_INDICADORES)
linha_cab_ind = localizar_linha_cabecalho(bruto_indicadores)
linha_dados_ind = localizar_inicio_dos_dados(bruto_indicadores, a_partir_de=linha_cab_ind)

print(f"Aba '{ABA_INDICADORES}':")
print(f"  -> Cabeçalho de colunas na linha {linha_cab_ind} (0-based) = linha {linha_cab_ind + 1} do Excel")
print(f"  -> Dados começam na linha {linha_dados_ind} (0-based) = linha {linha_dados_ind + 1} do Excel")
print(f"  -> Total de linhas na aba: {len(bruto_indicadores)}")

Aba 'Indicadores por município':
  -> Cabeçalho de colunas na linha 7 (0-based) = linha 8 do Excel
  -> Dados começam na linha 11 (0-based) = linha 12 do Excel
  -> Total de linhas na aba: 5581


### 3.2 Localizando as informações-alvo

Este arquivo já traz indicadores calculados pelo governo — cada coluna tem,
logo abaixo do nome, a fórmula usada (em termos dos códigos brutos do
formulário técnico) e o código do próprio indicador (ex.: `IGR0001`). Isso já
serve como validação própria, sem precisar do glossário do arquivo 1.

In [12]:
nomes_colunas_ind = bruto_indicadores.iloc[linha_cab_ind].tolist()

candidatas_critico_ind = buscar_colunas(nomes_colunas_ind, "crítico")
candidatas_drenagem_vias = buscar_colunas(nomes_colunas_ind, "vias", "rede")
candidatas_investimento = buscar_colunas(nomes_colunas_ind, "investimento", "drenagem")
candidatas_domicilios_risco_pct = buscar_colunas(nomes_colunas_ind, "domic", "risco", "inunda")

# As 2 linhas seguintes ao cabeçalho trazem fórmula e unidade; o código do indicador
# vem 3 linhas abaixo do cabeçalho (mesma lógica de sanity-check usada no arquivo 1)
linha_formula_ind = linha_cab_ind + 1
linha_unidade_ind = linha_cab_ind + 2
linha_codigo_ind = linha_cab_ind + 3

formulas_ind = bruto_indicadores.iloc[linha_formula_ind].tolist()
unidades_ind = bruto_indicadores.iloc[linha_unidade_ind].tolist()
codigos_ind = bruto_indicadores.iloc[linha_codigo_ind].tolist()

# valida que a linha de código realmente parece códigos de indicador (ex.: IGR0001)
padrao_codigo_indicador = re.compile(r"^[A-Z]{2,4}\d{3,4}\*?$")
amostra_valores = [v for v in codigos_ind if isinstance(v, str)]
proporcao_valida = sum(1 for v in amostra_valores if padrao_codigo_indicador.match(v.strip())) / max(len(amostra_valores), 1)
print(f"Linha {linha_codigo_ind} parece ser a linha de códigos de indicador? "
      f"{proporcao_valida:.0%} dos valores batem com o padrão esperado.\n")

for etiqueta, candidatas in [
    ("município crítico", candidatas_critico_ind),
    ("% vias com rede de drenagem", candidatas_drenagem_vias),
    ("investimento per capita em drenagem", candidatas_investimento),
    ("% domicílios em risco de inundação", candidatas_domicilios_risco_pct),
]:
    print(f"--- {etiqueta} ---")
    for idx, nome in candidatas:
        print(f"  col {idx} | código: {codigos_ind[idx]!r} | fórmula: {formulas_ind[idx]!r} | unidade: {unidades_ind[idx]!r}")
        print(f"           nome completo: {nome!r}")
    print()

Linha 10 parece ser a linha de códigos de indicador? 76% dos valores batem com o padrão esperado.

--- município crítico ---
  col 5 | código: '-' | fórmula: None | unidade: '-'
           nome completo: 'Crítico'

--- % vias com rede de drenagem ---
  col 25 | código: 'IAP0002' | fórmula: '(GAP0305 / GAP0304) x 100' | unidade: 'percentual'
           nome completo: 'Parcela de vias públicas com redes de águas pluviais subterrâneas na área urbana'
  col 31 | código: 'IAP0008' | fórmula: 'GAP0306 / GAP0305' | unidade: 'unidades/km'
           nome completo: 'Quantidade média de pontos de captação de águas pluviais em vias públicas urbanas com redes de águas pluviais'
  col 32 | código: 'IAP0009' | fórmula: 'GAP0312 / GAP0305' | unidade: 'unidades/km'
           nome completo: 'Quantidade média de poços de visita (PV) em vias públicas urbanas com redes de águas pluviais'

--- investimento per capita em drenagem ---
  col 23 | código: 'IFD0010' | fórmula: 'GFI2321 / DFE0002 ' | unidade: '

### 3.3 Amostra de 5 linhas — Arquivo 2 (aba de indicadores)

In [13]:
colunas_interesse_ind = [0, 1, 2]
for candidatas in [candidatas_critico_ind, candidatas_drenagem_vias, candidatas_investimento, candidatas_domicilios_risco_pct]:
    colunas_interesse_ind += [idx for idx, _ in candidatas]
colunas_interesse_ind = sorted(set(colunas_interesse_ind))

amostra_ind = montar_amostra(bruto_indicadores, linha_cab_ind, linha_dados_ind, n_linhas=5)
amostra_ind.iloc[:, colunas_interesse_ind]

,Código IBGE,Nome,UF,Crítico,Investimento total médio per capita nos serviços de Drenagem e Manejo de Águas Pluviais Urbanas,Parcela de vias públicas com redes de águas pluviais subterrâneas na área urbana,Quantidade média de pontos de captação de águas pluviais em vias públicas urbanas com redes de águas pluviais,Quantidade média de poços de visita (PV) em vias públicas urbanas com redes de águas pluviais,Parcela de domicílios sujeitos a risco de inundação na área urbana
0,1100015,Alta Floresta D Oeste,RO,Não,0,27.12,26.88,3.56,2.27
1,1100023,Ariquemes,RO,Sim,5.08,32.28,26.33,10.38,0.13
2,1100031,Cabixi,RO,Não,0,Não Calc.- Dados Não Inf.,Não Calc.- Dados Não Inf.,Não Calc.- Dados Não Inf.,0
3,1100049,Cacoal,RO,Sim,34.27,22.66,8.67,2.06,1.14
4,1100056,Cerejeiras,RO,Não,13.95,10.29,33.52,0.96,0


## 4. Resumo do diagnóstico (colunas confirmadas)

| Informação | Arquivo | Aba | Linha do cabeçalho (Excel) | Linha de início dos dados (Excel) | Coluna encontrada |
|---|---|---|---|---|---|
| Domicílios em risco de inundação (quantidade) — **GAP2110**, não GAP2109 | 1 | Dados sobre Gestão de Riscos | ver `linha_cab_riscos+1` acima | ver `linha_dados_riscos+1` acima | ver `candidatas_quantidade` |
| Existe mapeamento de área de risco — **GAP2105** | 1 | Dados sobre Gestão de Riscos | idem | idem | ver `candidatas_mapeamento` |
| Município crítico | 2 | Indicadores por município | ver `linha_cab_ind+1` acima | ver `linha_dados_ind+1` acima | ver `candidatas_critico_ind` |
| % vias com rede de drenagem — **IAP0002** | 2 | Indicadores por município | idem | idem | ver `candidatas_drenagem_vias` |
| Investimento per capita em drenagem — **IFD0010** | 2 | Indicadores por município | idem | idem | ver `candidatas_investimento` |
| % domicílios em risco (indicador pronto) — **IGR0001**, mantido junto com o valor absoluto do GAP2110 | 2 | Indicadores por município | idem | idem | ver `candidatas_domicilios_risco_pct` |

**Colunas validadas.** Cada uma foi selecionada por código exato (ou, no caso
de `Crítico`, por conferência da definição de GAP2001 no glossário) — não por
posição ou nome aproximado. O notebook segue abaixo para a limpeza
estrutural, a padronização do `Cod_IBGE_7` e a exportação do `sinisa_limpo.csv`.

## 5. Etapa 2 — Limpeza estrutural

Aqui isolamos so as colunas confirmadas, padronizamos o codigo do municipio e
juntamos os dois arquivos. Nenhuma regra de negocio e decidida nesta etapa —
os casos de domicilios "0 em risco" e os valores vazios sao apenas contados e
documentados, nao tratados.

In [14]:
log_transformacoes = []

def registrar_passo(nome, linhas_antes, linhas_depois, motivo):
    log_transformacoes.append({
        "Etapa": nome,
        "Linhas antes": linhas_antes,
        "Linhas depois": linhas_depois,
        "Motivo": motivo,
    })
    print(f"[{nome}] {linhas_antes} -> {linhas_depois} linhas. Motivo: {motivo}")

### 5.1 Selecionando as colunas confirmadas por codigo (nao por posicao)

Em vez de usar indice de coluna fixo (fragil se alguem reordenar colunas na
planilha), selecionamos cada coluna pelo codigo oficial do SINISA que ja
identificamos e voce confirmou.

In [15]:
def selecionar_por_codigo(candidatas, codigos, codigo_alvo):
    """Dentre as colunas candidatas (lista de (idx, nome)), retorna a que tem
    o codigo exato pedido. Levanta erro claro se nao achar - preferimos falhar
    a adivinhar."""
    for idx, nome in candidatas:
        codigo_col = codigos[idx]
        if isinstance(codigo_col, str) and codigo_col.strip().rstrip("*") == codigo_alvo:
            return idx, nome
    raise ValueError(
        f"Nao encontrei nenhuma coluna candidata com o codigo '{codigo_alvo}'. "
        f"Candidatas disponiveis: {[(i, codigos[i]) for i, _ in candidatas]}"
    )

# --- Arquivo 1: mapeamento (GAP2105) e quantidade de domicilios em risco (GAP2110) ---
idx_mapeamento, nome_mapeamento = selecionar_por_codigo(candidatas_mapeamento, codigos_riscos, "GAP2105")
idx_qtd_domicilios, nome_qtd_domicilios = selecionar_por_codigo(candidatas_quantidade, codigos_riscos, "GAP2110")

print(f"Mapeamento de risco -> col {idx_mapeamento}: {nome_mapeamento!r} (GAP2105)")
print(f"Quantidade de domicilios em risco -> col {idx_qtd_domicilios}: {nome_qtd_domicilios!r} (GAP2110)")

# --- Arquivo 2: critico, % vias com drenagem (IAP0002), investimento per capita (IFD0010), % domicilios risco (IGR0001) ---
def selecionar_critico_validado_por_glossario(candidatas, texto_glossario, codigo_glossario="GAP2001"):
    """A coluna 'Critico' do arquivo 2 nao carrega um codigo de campo proprio na
    linha de codigos do indicador (fica '-' ali) - o codigo GAP2001 que a
    identifica vem do glossario do arquivo 1. Por isso a validacao aqui e por
    unicidade (erro se houver mais de 1 candidata) + checagem de que a definicao
    de GAP2001 no glossario realmente fala de municipio critico, em vez de so
    pegar a primeira candidata silenciosamente."""
    if len(candidatas) != 1:
        raise ValueError(
            f"Esperava exatamente 1 coluna candidata para 'critico', encontrei {len(candidatas)}: "
            f"{candidatas}. Ajuste a selecao antes de prosseguir - nao escolha a primeira silenciosamente."
        )
    idx, nome = candidatas[0]
    pos = texto_glossario.find(codigo_glossario)
    if pos == -1:
        raise ValueError(f"Codigo {codigo_glossario} nao encontrado no glossario - nao consigo validar a coluna 'Critico'.")
    trecho = texto_glossario[pos: pos + 300].lower()
    if "crítico" not in trecho and "critico" not in trecho:
        raise ValueError(
            f"A definicao de {codigo_glossario} no glossario nao menciona 'critico' - "
            "a coluna pode nao ser a esperada."
        )
    return idx, nome

idx_critico, nome_critico = selecionar_critico_validado_por_glossario(candidatas_critico_ind, texto_glossario)
idx_pct_vias, nome_pct_vias = selecionar_por_codigo(candidatas_drenagem_vias, codigos_ind, "IAP0002")
idx_investimento, nome_investimento = selecionar_por_codigo(candidatas_investimento, codigos_ind, "IFD0010")
idx_pct_domicilios, nome_pct_domicilios = selecionar_por_codigo(candidatas_domicilios_risco_pct, codigos_ind, "IGR0001")

print(f"\nCritico -> col {idx_critico}: {nome_critico!r}")
print(f"% vias com rede de drenagem -> col {idx_pct_vias}: {nome_pct_vias!r} (IAP0002)")
print(f"Investimento per capita em drenagem -> col {idx_investimento}: {nome_investimento!r} (IFD0010)")
print(f"% domicilios em risco (indicador pronto) -> col {idx_pct_domicilios}: {nome_pct_domicilios!r} (IGR0001)")

Mapeamento de risco -> col 34: "Existe mapeamento de áreas de risco de inundação dos cursos d'água urbanos? *" (GAP2105)
Quantidade de domicilios em risco -> col 39: 'Quantidade de domicílios sujeitos a risco de inundação. *' (GAP2110)

Critico -> col 5: 'Crítico'
% vias com rede de drenagem -> col 25: 'Parcela de vias públicas com redes de águas pluviais subterrâneas na área urbana' (IAP0002)
Investimento per capita em drenagem -> col 23: 'Investimento total médio per capita nos serviços de Drenagem e Manejo de Águas Pluviais Urbanas' (IFD0010)
% domicilios em risco (indicador pronto) -> col 33: 'Parcela de domicílios sujeitos a risco de inundação na área urbana' (IGR0001)


### 5.2 Extraindo apenas as colunas relevantes de cada arquivo

In [16]:
# Todas as linhas de dados de cada aba (sem cabecalho ainda)
dados_riscos_completo = bruto_riscos.iloc[linha_dados_riscos:].reset_index(drop=True)
dados_ind_completo = bruto_indicadores.iloc[linha_dados_ind:].reset_index(drop=True)

registrar_passo(
    "Carregar dados brutos (arquivo 1, aba Gestao de Riscos)",
    linhas_antes=len(bruto_riscos),
    linhas_depois=len(dados_riscos_completo),
    motivo="Descartadas as linhas de titulo/cabecalho/formula/unidade/codigo acima do inicio real dos dados."
)
registrar_passo(
    "Carregar dados brutos (arquivo 2, Indicadores)",
    linhas_antes=len(bruto_indicadores),
    linhas_depois=len(dados_ind_completo),
    motivo="Descartadas as linhas de titulo/cabecalho/formula/unidade/codigo acima do inicio real dos dados."
)

df1 = dados_riscos_completo.iloc[:, [0, 1, 2, idx_mapeamento, idx_qtd_domicilios]].copy()
df1.columns = ["Cod_IBGE_7", "Nome", "UF", "mapeamento_area_risco", "qtd_domicilios_risco"]

df2 = dados_ind_completo.iloc[:, [0, 1, 2, idx_critico, idx_pct_vias, idx_investimento, idx_pct_domicilios]].copy()
df2.columns = ["Cod_IBGE_7", "Nome", "UF", "municipio_critico", "pct_vias_rede_drenagem", "investimento_percapita_drenagem", "pct_domicilios_risco"]

print("df1 (arquivo 1, colunas isoladas):", df1.shape)
display(df1.head())
print("\ndf2 (arquivo 2, colunas isoladas):", df2.shape)
display(df2.head())

[Carregar dados brutos (arquivo 1, aba Gestao de Riscos)] 5581 -> 5570 linhas. Motivo: Descartadas as linhas de titulo/cabecalho/formula/unidade/codigo acima do inicio real dos dados.
[Carregar dados brutos (arquivo 2, Indicadores)] 5581 -> 5570 linhas. Motivo: Descartadas as linhas de titulo/cabecalho/formula/unidade/codigo acima do inicio real dos dados.
df1 (arquivo 1, colunas isoladas): (5570, 5)


,Cod_IBGE_7,Nome,UF,mapeamento_area_risco,qtd_domicilios_risco
0,1100015,Alta Floresta D Oeste,RO,Sim,150
1,1100023,Ariquemes,RO,Sim,53
2,1100031,Cabixi,RO,Não,0
3,1100049,Cacoal,RO,Sim,430
4,1100056,Cerejeiras,RO,Não,0



df2 (arquivo 2, colunas isoladas): (5570, 7)


,Cod_IBGE_7,Nome,UF,municipio_critico,pct_vias_rede_drenagem,investimento_percapita_drenagem,pct_domicilios_risco
0,1100015,Alta Floresta D Oeste,RO,Não,27.12,0,2.27
1,1100023,Ariquemes,RO,Sim,32.28,5.08,0.13
2,1100031,Cabixi,RO,Não,Não Calc.- Dados Não Inf.,0,0
3,1100049,Cacoal,RO,Sim,22.66,34.27,1.14
4,1100056,Cerejeiras,RO,Não,10.29,13.95,0


### 5.3 Padronizando `Cod_IBGE_7` como inteiro de 7 digitos nos dois arquivos

In [17]:
for nome_df, df in [("df1", df1), ("df2", df2)]:
    antes = len(df)
    cod_numerico = pd.to_numeric(df["Cod_IBGE_7"], errors="coerce")
    invalidos = cod_numerico.isna().sum()
    df["Cod_IBGE_7"] = cod_numerico

    tamanhos = df["Cod_IBGE_7"].dropna().astype("int64").astype(str).str.len()
    fora_do_padrao = (tamanhos != 7).sum()

    print(f"{nome_df}: {invalidos} valor(es) de Cod_IBGE_7 nao numerico(s); "
          f"{fora_do_padrao} valor(es) numerico(s) mas fora do padrao de 7 digitos.")

df1: 0 valor(es) de Cod_IBGE_7 nao numerico(s); 0 valor(es) numerico(s) mas fora do padrao de 7 digitos.
df2: 0 valor(es) de Cod_IBGE_7 nao numerico(s); 0 valor(es) numerico(s) mas fora do padrao de 7 digitos.


In [18]:
# Converte para Int64 nullable (permite manter NaN sem virar float com casas decimais)
df1["Cod_IBGE_7"] = df1["Cod_IBGE_7"].astype("Int64")
df2["Cod_IBGE_7"] = df2["Cod_IBGE_7"].astype("Int64")

print("dtype Cod_IBGE_7 em df1:", df1["Cod_IBGE_7"].dtype)
print("dtype Cod_IBGE_7 em df2:", df2["Cod_IBGE_7"].dtype)

dtype Cod_IBGE_7 em df1: Int64
dtype Cod_IBGE_7 em df2: Int64


### 5.4 Removendo linhas sem codigo de municipio (rodapes, totais, linhas em branco)

In [19]:
antes_df1 = len(df1)
df1 = df1.dropna(subset=["Cod_IBGE_7"]).reset_index(drop=True)
registrar_passo(
    "dropna em Cod_IBGE_7 (arquivo 1)",
    linhas_antes=antes_df1,
    linhas_depois=len(df1),
    motivo="Remove linhas sem codigo de municipio (rodapes, totais, linhas em branco)."
)

antes_df2 = len(df2)
df2 = df2.dropna(subset=["Cod_IBGE_7"]).reset_index(drop=True)
registrar_passo(
    "dropna em Cod_IBGE_7 (arquivo 2)",
    linhas_antes=antes_df2,
    linhas_depois=len(df2),
    motivo="Remove linhas sem codigo de municipio (rodapes, totais, linhas em branco)."
)

[dropna em Cod_IBGE_7 (arquivo 1)] 5570 -> 5570 linhas. Motivo: Remove linhas sem codigo de municipio (rodapes, totais, linhas em branco).
[dropna em Cod_IBGE_7 (arquivo 2)] 5570 -> 5570 linhas. Motivo: Remove linhas sem codigo de municipio (rodapes, totais, linhas em branco).


### 5.5 Tratando a string literal "null" nas colunas categoricas

Ao inspecionar o Excel bruto, os 725 municipios que nao participaram da
pesquisa (os mesmos que ficam sem valor em `qtd_domicilios_risco`) tem, na
coluna de mapeamento de risco, a **string literal `"null"`** em vez de uma
celula realmente vazia - por isso `.isna()` nao pegava esses casos antes.
Aqui convertemos essa string (e variacoes de maiusculas/minusculas e espacos
em volta) para um valor ausente de verdade, em `mapeamento_area_risco` e,
defensivamente, tambem em `municipio_critico` (mesmo que hoje ela nao
apresente o problema).

In [20]:
def relatar_valores(df, coluna, nome_df, valores_esperados=("Sim", "Não")):
    valores_esperados = ("Sim", "Não")
    contagem = df[coluna].value_counts(dropna=False)
    print(f"--- {nome_df}.{coluna} ---")
    print(contagem)
    inesperados = [v for v in contagem.index if pd.notna(v) and v not in ("Sim", "Não")]
    if inesperados:
        print(f"  Valor(es) fora do padrao esperado (Sim/Não): {inesperados}")
    else:
        print(f"  Nenhum valor fora do padrao esperado (Sim/Não), alem de nulo.")
    print()
    return contagem

def limpar_null_literal(serie):
    def normalizar(v):
        if isinstance(v, str) and v.strip().lower() == "null":
            return pd.NA
        return v
    return serie.apply(normalizar)

print("### ANTES da limpeza ###\n")
relatar_valores(df1, "mapeamento_area_risco", "df1")
relatar_valores(df2, "municipio_critico", "df2")

for nome_df, df, coluna in [("df1", df1, "mapeamento_area_risco"), ("df2", df2, "municipio_critico")]:
    qtd_null_literal = df[coluna].apply(lambda v: isinstance(v, str) and v.strip().lower() == "null").sum()
    df[coluna] = limpar_null_literal(df[coluna])
    print(f"{nome_df}.{coluna}: {qtd_null_literal} string(s) 'null' literal convertida(s) para nulo de verdade.")

print("\n### DEPOIS da limpeza ###\n")
relatar_valores(df1, "mapeamento_area_risco", "df1")
relatar_valores(df2, "municipio_critico", "df2")

### ANTES da limpeza ###

--- df1.mapeamento_area_risco ---
mapeamento_area_risco
Não     3385
Sim     1460
null     725
Name: count, dtype: int64
  Valor(es) fora do padrao esperado (Sim/Não): ['null']

--- df2.municipio_critico ---
municipio_critico
Não    3634
Sim    1936
Name: count, dtype: int64
  Nenhum valor fora do padrao esperado (Sim/Não), alem de nulo.

df1.mapeamento_area_risco: 725 string(s) 'null' literal convertida(s) para nulo de verdade.
df2.municipio_critico: 0 string(s) 'null' literal convertida(s) para nulo de verdade.

### DEPOIS da limpeza ###

--- df1.mapeamento_area_risco ---
mapeamento_area_risco
Não     3385
Sim     1460
<NA>     725
Name: count, dtype: int64
  Nenhum valor fora do padrao esperado (Sim/Não), alem de nulo.

--- df2.municipio_critico ---
municipio_critico
Não    3634
Sim    1936
Name: count, dtype: int64
  Nenhum valor fora do padrao esperado (Sim/Não), alem de nulo.



,count
municipio_critico,
Não,3634
Sim,1936


### 5.6 Convertendo as colunas numericas e contando quantas linhas foram afetadas

`municipio_critico` e `mapeamento_area_risco` sao categoricas (Sim/Nao) e
ficam como estao — so as colunas realmente numericas passam por
`pd.to_numeric(errors='coerce')`.

In [21]:
colunas_numericas_df1 = ["qtd_domicilios_risco"]
colunas_numericas_df2 = ["pct_vias_rede_drenagem", "investimento_percapita_drenagem", "pct_domicilios_risco"]

def converter_numericas(df, colunas, nome_df):
    for col in colunas:
        valores_antes_nulos = df[col].isna().sum()
        convertido = pd.to_numeric(df[col], errors="coerce")
        novos_nulos = convertido.isna().sum() - valores_antes_nulos
        df[col] = convertido
        print(f"{nome_df}.{col}: {novos_nulos} linha(s) tinham valor nao numerico "
              f"(ex.: '-', texto, espaco em branco) e viraram NaN na conversao.")

converter_numericas(df1, colunas_numericas_df1, "df1")
converter_numericas(df2, colunas_numericas_df2, "df2")

df1.qtd_domicilios_risco: 725 linha(s) tinham valor nao numerico (ex.: '-', texto, espaco em branco) e viraram NaN na conversao.
df2.pct_vias_rede_drenagem: 2102 linha(s) tinham valor nao numerico (ex.: '-', texto, espaco em branco) e viraram NaN na conversao.
df2.investimento_percapita_drenagem: 1560 linha(s) tinham valor nao numerico (ex.: '-', texto, espaco em branco) e viraram NaN na conversao.
df2.pct_domicilios_risco: 725 linha(s) tinham valor nao numerico (ex.: '-', texto, espaco em branco) e viraram NaN na conversao.


### 5.7 Percentual de preenchimento de cada coluna extraida

In [22]:
def percentual_preenchimento(df, colunas, nome_df):
    linhas = []
    for col in colunas:
        preenchido = df[col].notna().mean() * 100
        linhas.append({"Arquivo": nome_df, "Coluna": col, "% preenchido": round(preenchido, 1)})
    return linhas

resumo_preenchimento = (
    percentual_preenchimento(df1, ["mapeamento_area_risco", "qtd_domicilios_risco"], "df1")
    + percentual_preenchimento(df2, ["municipio_critico", "pct_vias_rede_drenagem", "investimento_percapita_drenagem", "pct_domicilios_risco"], "df2")
)

df_preenchimento = pd.DataFrame(resumo_preenchimento)
display(df_preenchimento)

,Arquivo,Coluna,% preenchido
0,df1,mapeamento_area_risco,87.0
1,df1,qtd_domicilios_risco,87.0
2,df2,municipio_critico,100.0
3,df2,pct_vias_rede_drenagem,62.3
4,df2,investimento_percapita_drenagem,72.0
5,df2,pct_domicilios_risco,87.0


### 5.8 Documentando (sem decidir) os casos de "0 domicilios em risco" e valores vazios

Isso e regra de negocio ainda nao fechada pela squad — aqui so contamos, nao
tratamos.

In [23]:
qtd_zero = (df1["qtd_domicilios_risco"] == 0).sum()
qtd_vazio_df1 = df1["qtd_domicilios_risco"].isna().sum()
qtd_vazio_mapeamento = df1["mapeamento_area_risco"].isna().sum()

print(f"df1: {qtd_zero} municipio(s) declaram 0 domicilios em risco de inundacao (GAP2110).")
print(f"df1: {qtd_vazio_df1} municipio(s) sem valor para domicilios em risco (vazio).")
print(f"df1: {qtd_vazio_mapeamento} municipio(s) sem resposta para 'existe mapeamento de area de risco'.")

for col in ["municipio_critico", "pct_vias_rede_drenagem", "investimento_percapita_drenagem", "pct_domicilios_risco"]:
    vazios = df2[col].isna().sum()
    print(f"df2.{col}: {vazios} valor(es) vazio(s).")

print("\nNenhuma decisao foi tomada sobre esses casos (nao foram removidos, zerados ou imputados) - "
      "fica para a squad decidir a regra de negocio.")

df1: 2607 municipio(s) declaram 0 domicilios em risco de inundacao (GAP2110).
df1: 725 municipio(s) sem valor para domicilios em risco (vazio).
df1: 725 municipio(s) sem resposta para 'existe mapeamento de area de risco'.
df2.municipio_critico: 0 valor(es) vazio(s).
df2.pct_vias_rede_drenagem: 2102 valor(es) vazio(s).
df2.investimento_percapita_drenagem: 1560 valor(es) vazio(s).
df2.pct_domicilios_risco: 725 valor(es) vazio(s).

Nenhuma decisao foi tomada sobre esses casos (nao foram removidos, zerados ou imputados) - fica para a squad decidir a regra de negocio.


### 5.9 Juncao dos dois arquivos por `Cod_IBGE_7`

Juncao externa (`outer`): mantem municipios mesmo quando so aparecem em um dos
dois arquivos.

In [24]:
codigos_df1 = set(df1["Cod_IBGE_7"])
codigos_df2 = set(df2["Cod_IBGE_7"])

so_df1 = codigos_df1 - codigos_df2
so_df2 = codigos_df2 - codigos_df1
em_ambos = codigos_df1 & codigos_df2

print(f"Municipios so no arquivo 1: {len(so_df1)}")
print(f"Municipios so no arquivo 2: {len(so_df2)}")
print(f"Municipios em ambos: {len(em_ambos)}")

df_final = pd.merge(
    df1,
    df2.drop(columns=["Nome", "UF"]),  # ja vem de df1, evita colunas duplicadas
    on="Cod_IBGE_7",
    how="outer",
    validate="one_to_one",
)

registrar_passo(
    "Merge df1 + df2 por Cod_IBGE_7 (outer)",
    linhas_antes=f"{len(df1)} (df1) + {len(df2)} (df2)",
    linhas_depois=len(df_final),
    motivo="Juncao externa para nao perder municipio que so apareceu em um dos dois arquivos."
)

print(f"\nLinhas no dataframe final: {len(df_final)}")
display(df_final.head())

Municipios so no arquivo 1: 0
Municipios so no arquivo 2: 0
Municipios em ambos: 5570
[Merge df1 + df2 por Cod_IBGE_7 (outer)] 5570 (df1) + 5570 (df2) -> 5570 linhas. Motivo: Juncao externa para nao perder municipio que so apareceu em um dos dois arquivos.

Linhas no dataframe final: 5570


,Cod_IBGE_7,Nome,UF,mapeamento_area_risco,qtd_domicilios_risco,municipio_critico,pct_vias_rede_drenagem,investimento_percapita_drenagem,pct_domicilios_risco
0,1100015,Alta Floresta D Oeste,RO,Sim,150.0,Não,27.12,0.00,2.27
1,1100023,Ariquemes,RO,Sim,53.0,Sim,32.28,5.08,0.13
2,1100031,Cabixi,RO,Não,0.0,Não,NaN,0.00,0.00
3,1100049,Cacoal,RO,Sim,430.0,Sim,22.66,34.27,1.14
4,1100056,Cerejeiras,RO,Não,0.0,Não,10.29,13.95,0.00


## 6. Etapa 3 — Documentacao obrigatoria (protocolo DataConnect)

### 6.a Log de transformacoes

Gerado dinamicamente a partir de cada passo executado acima (numero de linhas
antes/depois e o motivo de cada decisao).

In [25]:
df_log = pd.DataFrame(log_transformacoes)
display(df_log)

,Etapa,Linhas antes,Linhas depois,Motivo
0,"Carregar dados brutos (arquivo 1, aba Gestao de Riscos)",5581,5570,Descartadas as linhas de titulo/cabecalho/formula/unidade/codigo acima do inicio real dos dados.
1,"Carregar dados brutos (arquivo 2, Indicadores)",5581,5570,Descartadas as linhas de titulo/cabecalho/formula/unidade/codigo acima do inicio real dos dados.
2,dropna em Cod_IBGE_7 (arquivo 1),5570,5570,"Remove linhas sem codigo de municipio (rodapes, totais, linhas em branco)."
3,dropna em Cod_IBGE_7 (arquivo 2),5570,5570,"Remove linhas sem codigo de municipio (rodapes, totais, linhas em branco)."
4,Merge df1 + df2 por Cod_IBGE_7 (outer),5570 (df1) + 5570 (df2),5570,Juncao externa para nao perder municipio que so apareceu em um dos dois arquivos.


In [26]:
from IPython.display import display, Markdown

linhas_md = ["| Etapa | Linhas antes | Linhas depois | Motivo |", "|---|---|---|---|"]
for passo in log_transformacoes:
    linhas_md.append(f"| {passo['Etapa']} | {passo['Linhas antes']} | {passo['Linhas depois']} | {passo['Motivo']} |")

display(Markdown("\n".join(linhas_md)))

| Etapa | Linhas antes | Linhas depois | Motivo |
|---|---|---|---|
| Carregar dados brutos (arquivo 1, aba Gestao de Riscos) | 5581 | 5570 | Descartadas as linhas de titulo/cabecalho/formula/unidade/codigo acima do inicio real dos dados. |
| Carregar dados brutos (arquivo 2, Indicadores) | 5581 | 5570 | Descartadas as linhas de titulo/cabecalho/formula/unidade/codigo acima do inicio real dos dados. |
| dropna em Cod_IBGE_7 (arquivo 1) | 5570 | 5570 | Remove linhas sem codigo de municipio (rodapes, totais, linhas em branco). |
| dropna em Cod_IBGE_7 (arquivo 2) | 5570 | 5570 | Remove linhas sem codigo de municipio (rodapes, totais, linhas em branco). |
| Merge df1 + df2 por Cod_IBGE_7 (outer) | 5570 (df1) + 5570 (df2) | 5570 | Juncao externa para nao perder municipio que so apareceu em um dos dois arquivos. |

## 7. Etapa 4 — Exportacao

In [27]:
CAMINHO_SAIDA = "sinisa_limpo.csv"
df_final.to_csv(CAMINHO_SAIDA, index=False, encoding="utf-8-sig")

print(f"Exportado: {CAMINHO_SAIDA}")
print(f"Linhas: {len(df_final)} | Colunas: {list(df_final.columns)}")

try:
    from google.colab import files as colab_files
    colab_files.download(CAMINHO_SAIDA)
except ImportError:
    print("Fora do Colab: o arquivo foi salvo no diretorio atual do notebook.")

Exportado: sinisa_limpo.csv
Linhas: 5570 | Colunas: ['Cod_IBGE_7', 'Nome', 'UF', 'mapeamento_area_risco', 'qtd_domicilios_risco', 'municipio_critico', 'pct_vias_rede_drenagem', 'investimento_percapita_drenagem', 'pct_domicilios_risco']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>